# 03: Double Pulse Transients - Golden vs Tier-2 vs Tier-3

Clamped-inductive two-pulse test. Every waveform is **measured**: the golden
DPT trajectory is captured by the pinned bench at
`simulation/testbenches/dpt_trajectory.py`, and the Tier-2 / Tier-3 ROMs are
simulated with the standard `run_double_pulse` bench on the emitted
subcircuits. Timing/energy and waveform RMSE come from the measured JSONs in
`reports/comparison/<vendor>/`. Golden-vs-Tier-2 figures are written to
`figures/<vendor>/tier2/`, golden-vs-Tier-3 to `figures/<vendor>/tier3/`
(identical base names). No synthetic waveforms and no invented speedups;
absent level crossings stay `not available`.

In [1]:
import sys, os, json
from pathlib import Path
os.environ.setdefault('MPLBACKEND', 'Agg')
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT)); os.chdir(REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from common.plotting import setup_style, plot_waveform_comparison
from simulation.testbenches.double_pulse import run_double_pulse
from simulation.testbenches.dpt_trajectory import capture_trajectory  # pinned path

setup_style()

VENDORS = ('irfs7730pbf', 'csd16412q5a', 'epc2216', 'cpm2-1200')
CMP = REPO_ROOT / 'reports' / 'comparison'
MODELS = REPO_ROOT / 'reports' / 'models'
METRIC_KEYS = ('td_on_ns', 'tr_ns', 'td_off_ns', 'tf_ns',
               'dv_dt_max_v_per_ns', 'di_dt_max_a_per_ns',
               'e_on_uj', 'e_off_uj', 'i_peak_a')

In [2]:
def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def disp(v):
    """Measured value, or an explicit 'not available' cell (never imputed)."""
    if v is None:
        return 'not available'
    try:
        f = float(v)
    except (TypeError, ValueError):
        return str(v)
    return 'not available' if not np.isfinite(f) else f'{f:.4g}'


def pct(v):
    if v is None:
        return 'not available'
    f = float(v)
    return 'not available' if not np.isfinite(f) else f'{f:+.2f}%'


def emitted_model(lib):
    """Read (subcircuit name, pin roles) from an emitted lib's .subckt header.

    Tier-3 emitters keep their own subckt names and vendor port order
    (EPC is gate-first; CPM2 has a 4th thermal 'tj' port), so neither the
    filename stem nor a fixed pin list is safe.
    """
    text = lib.read_text(encoding='ascii', errors='ignore')
    header = None
    for line in text.splitlines():
        s = line.strip()
        if s.lower().startswith('.subckt '):
            header = s.split()
            break
    if header is None:
        raise ValueError(f'no .subckt header in {lib.name}')
    ports = [p.lower() for p in header[2:] if '=' not in p]
    role = {'d': 'drain', 'g': 'gate', 's': 'source',
            'drain': 'drain', 'gate': 'gate', 'source': 'source',
            'tj': 'tj'}
    return header[1], [role.get(port, port) for port in ports]


def measure(fn, *args, **kwargs):
    """Re-run a transient measurement on transient LTspice failures only.

    No data is substituted: a failed solver run is retried, and a persistent
    failure is raised so the notebook reports it rather than inventing output.
    """
    import time
    for attempt in range(3):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            tail = getattr(exc, 'log_tail', '') or ''
            if attempt == 2:
                raise
            print(f'  transient LTspice failure ({type(exc).__name__}); '
                  f'retry {attempt + 1}/2. log tail: {tail[-300:]!r}', flush=True)
            time.sleep(2)


In [3]:
metric_rows, run_rows = [], []
for vendor in VENDORS:
    print(f'{vendor}: starting measured DPT runs', flush=True)
    cfg = yaml.safe_load((REPO_ROOT / 'config' / 'inputs' / f'{vendor}.yaml').read_text(encoding='utf-8'))
    dpt = cfg['dpt']
    args = {k: dpt[k] for k in ('vbus', 'vgs_on', 'i_load_target', 'l_load',
                                't1_on', 't1_width', 't2_on', 't2_width',
                                'edge_time', 't_end', 'max_step')}
    temp = cfg['conditions'].get('temperature', 25.0)

    # Golden DPT trajectory (measured). Trajectory.input = [Vgs, Vds],
    # Trajectory.output = [Id, Ig]; source is grounded so Vds = V(drain).
    gold = measure(capture_trajectory, vendor)
    t_g = gold.time * 1e9
    g_vds, g_id, g_vgs = gold.input[1], gold.output[0], gold.input[0]

    # Emitted ROMs simulated with the same bench and conditions.
    rom = {}
    for tier, tag in (('tier2', 'T2'), ('tier3', 'T3')):
        lib = MODELS / f'{vendor}_PIECESPICE_{tag}.lib'
        sub, pins = emitted_model(lib)
        r = measure(run_double_pulse, lib, sub, pins,
                    temperature=temp, **args)
        rom[tier] = {'t': r.time * 1e9, 'vds': r.v_ds, 'id': r.i_d, 'vgs': r.v_gs}
    # Golden vs Tier-2 -> tier2/, Golden vs Tier-3 -> tier3/.
    for tier, tag in (('tier2', 'Tier-2'), ('tier3', 'Tier-3')):
        r = rom[tier]
        plot_waveform_comparison(
            t_g,
            {'golden': g_vds, tag: np.interp(t_g, r['t'], r['vds'])},
            {'golden': g_id, tag: np.interp(t_g, r['t'], r['id'])},
            vendor=vendor, filename='dpt_waveforms.png', tier=tier,
            left_label='Vds [V]', right_label='Id [A]',
            title=f'{vendor}: DPT waveforms - golden vs {tag}')
        plot_waveform_comparison(
            t_g, {'golden': g_vgs, tag: np.interp(t_g, r['t'], r['vgs'])},
            {}, vendor=vendor, filename='dpt_gate.png', tier=tier,
            left_label='Vgs [V]',
            title=f'{vendor}: gate drive - golden vs {tag}',
            figsize=(10.5, 5.8))
    plt.close('all')

    v3 = load_json(CMP / vendor / 'tier3_validation.json')
    v2 = load_json(CMP / vendor / 'dpt_metrics.json')
    cmp = v3['dpt_comparison']
    for key in METRIC_KEYS:
        c = cmp[key]
        metric_rows.append({
            'vendor': vendor, 'metric': key,
            'golden': disp(c['golden']), 'tier2': disp(c['tier2']),
            'tier3': disp(c['tier3']),
            'tier2_rel_err': pct(c['tier2_rel_err_pct']),
            'tier3_rel_err': pct(c['tier3_rel_err_pct']),
        })
    wf, rt, dr = v3['dpt_run']['waveform'], v3['runtime'], v3['dpt_run']
    run_rows.append({
        'vendor': vendor,
        'golden_pts': dr['golden_trace_points'],
        'tier3_pts': dr['trace_points'],
        'tier3_iterations': dr['iterations'],
        'tier3_completed': dr['completed'],
        'Vds_RMSE': disp(wf['vds_rel_rmse_pct']) + '%',
        'Id_RMSE': disp(wf['id_rel_rmse_pct']) + '%',
        'golden_wall_s': disp(rt['golden']['wall_clock_s']),
        'tier3_wall_s': disp(rt['tier3']['wall_clock_s']),
        'golden_solver_s': disp(rt['golden']['solver_elapsed_s']),
        'tier3_solver_s': disp(rt['tier3']['solver_elapsed_s']),
        'solver_ratio': disp(rt['solver_speedup']),
        't2_golden_failure': v2.get('golden_failure') or '',
    })
    print(f'{vendor}: DPT figures written to tier2/ and tier3/; '
          f'Tier-3 golden pts={dr["golden_trace_points"]}, '
          f'tier3 pts={dr["trace_points"]}', flush=True)

irfs7730pbf: starting measured DPT runs


irfs7730pbf: DPT figures written to tier2/ and tier3/; Tier-3 golden pts=7114, tier3 pts=7210


csd16412q5a: starting measured DPT runs


csd16412q5a: DPT figures written to tier2/ and tier3/; Tier-3 golden pts=10189, tier3 pts=66240


epc2216: starting measured DPT runs


epc2216: DPT figures written to tier2/ and tier3/; Tier-3 golden pts=7418, tier3 pts=9301


cpm2-1200: starting measured DPT runs


cpm2-1200: DPT figures written to tier2/ and tier3/; Tier-3 golden pts=7154, tier3 pts=31436


## Results

Timing/energy values are read verbatim from the measured
`tier3_validation.json` (`dpt_comparison`) for all three models. `not available`
means the level crossing was absent in that run - IRFS Tier-3 has no
`td_on`, `td_off`, `e_on` or `e_off` crossing, and those cells are never
imputed. The solver ratio is `golden solver time / Tier-3 solver time`
measured on both completed benches; every value is below 1.0 (a slowdown),
so **no speedup is claimed**.

In [4]:
metrics_df = pd.DataFrame(metric_rows)
runs_df = pd.DataFrame(run_rows)
print('Timing / energy - golden vs Tier-2 vs Tier-3 (measured JSON values)')
display(metrics_df.pivot(index='metric', columns='vendor',
                         values=['golden', 'tier2', 'tier3']))
print('\nDPT waveform RMSE, trace counts and measured solver times')
display(runs_df.set_index('vendor'))
print('IRFS Tier-3 td_on/td_off/e_on/e_off are "not available" - absent crossings, never imputed.')
print('All measured solver ratios are < 1.0 (slowdown); no speedup is claimed.')

Timing / energy - golden vs Tier-2 vs Tier-3 (measured JSON values)


golden                                     tier2  \
vendor             cpm2-1200 csd16412q5a epc2216 irfs7730pbf cpm2-1200   
metric                                                                   
di_dt_max_a_per_ns    0.5957      0.6675    3.71      0.2269    0.3811   
dv_dt_max_v_per_ns     373.7       600.4    3556       35.93     374.9   
e_off_uj                56.7       17.97   15.63         597     57.04   
e_on_uj                5.619       1.063   1.091       157.4     6.109   
i_peak_a               26.82       13.43   13.24       138.5     26.73   
td_off_ns              14.03       12.44   6.954       62.18     18.24   
td_on_ns               10.25       7.389    8.28        32.3     11.78   
tf_ns                  4.139       1.662   1.227       26.54     8.789   
tr_ns                  4.703       1.096  0.8456       25.84     4.123   

                                                       tier3              \
vendor             csd16412q5a epc2216 irfs7730pbf cpm2-1200 csd16412q5a   
metric                                                                     
di_dt_max_a_per_ns      0.1011  0.1449       2.498    0.4946       1.008   
dv_dt_max_v_per_ns       158.2   175.8       587.5      1170        1244   
e_off_uj                 17.73   15.41       533.4     55.43       17.55   
e_on_uj                 0.7531   1.149       71.09     4.465       0.941   
i_peak_a                 13.44   13.23         137     26.65       13.35   
td_off_ns                15.01   7.049       17.13     7.764       11.89   
td_on_ns                 5.408   8.511       17.08     8.641       7.053   
tf_ns                    2.246   1.343       2.529      2.53      0.6886   
tr_ns                    1.486  0.8378       2.893     0.814      0.1064   

                                           
vendor             epc2216    irfs7730pbf  
metric                                     
di_dt_max_a_per_ns    1.35       0.005455  
dv_dt_max_v_per_ns   733.8      1.907e-05  
e_off_uj             15.25  not available  
e_on_uj               1.26  not available  
i_peak_a             13.07          99.49  
td_off_ns            4.736  not available  
td_on_ns             8.717  not available  
tf_ns                2.078          0.865  
tr_ns                1.524          3.171


DPT waveform RMSE, trace counts and measured solver times


,golden_pts,tier3_pts,tier3_iterations,tier3_completed,Vds_RMSE,Id_RMSE,golden_wall_s,tier3_wall_s,golden_solver_s,tier3_solver_s,solver_ratio,t2_golden_failure
vendor,,,,,,,,,,,,
irfs7730pbf,7114,7210,19530,True,39.59%,13.09%,1.138,2.771,0.447,2.073,0.2156,
csd16412q5a,10189,66240,394001,True,2.375%,0.4201%,1.275,9.995,0.557,8.35,0.06671,
epc2216,7418,9301,33664,True,5.292%,0.7572%,1.221,2.786,0.47,2.052,0.229,
cpm2-1200,7154,31436,189005,True,4.215%,0.3455%,1.277,6.412,0.532,5.185,0.1026,


IRFS Tier-3 td_on/td_off/e_on/e_off are "not available" - absent crossings, never imputed.
All measured solver ratios are < 1.0 (slowdown); no speedup is claimed.
